# R1-01 Research pack
Writes `research/sector_research.md`, `research/opportunities_risks.md`, `research/use_cases.md`.
Every number in the problems section is computed from `data/processed/` when you run the notebook (Shopzilla = proxy/benchmark for Spreekorb).
Cells: 1 Check, 2 Backup, 3 Create, 4 Verify, 5 Unchanged, 6 Undo (RUN_UNDO = False), 7 Git (print only).

In [ ]:
# 1 Check: right folder, input files exist
import os, re, glob, shutil, hashlib, datetime
import pandas as pd

# move to the project root if we start inside notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

RESULTS = []  # one True/False per check
def check(name, ok):
    RESULTS.append(bool(ok))
    print('PASS' if ok else 'FAIL', '|', name)

check('Folder is the project root (AGENTS.md and specs/ found)', os.path.exists('AGENTS.md') and os.path.isdir('specs'))
check('Spec specs/R1-01_research.md exists', os.path.exists('specs/R1-01_research.md'))
INPUTS = ['shopzilla_contacts_clean.csv', 'shopzilla_email_clean.csv', 'tobi_tuning_150.csv']
for f in INPUTS:
    check('Input exists: data/processed/' + f, os.path.exists('data/processed/' + f))
print('Checks so far:', sum(RESULTS), 'of', len(RESULTS), 'PASS')

In [ ]:
# 2 Backup: copy every file we will change to _backup/<timestamp>/
OUT = ['research/sector_research.md', 'research/opportunities_risks.md', 'research/use_cases.md']
STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP = os.path.join('_backup', STAMP)
EXISTED = [f for f in OUT if os.path.exists(f)]  # files that already exist
for f in EXISTED:
    os.makedirs(os.path.join(BACKUP, os.path.dirname(f)), exist_ok=True)
    shutil.copy2(f, os.path.join(BACKUP, f))
print('Backed up', len(EXISTED), 'existing file(s) to', BACKUP)
print('New files (no backup needed):', [f for f in OUT if f not in EXISTED])

# hash the files we must NOT change (explicit list only), used in cell 5
def sha(p):
    return hashlib.sha256(open(p, 'rb').read()).hexdigest()
def watched():
    files = glob.glob('data/processed/*.csv') + ['data/README.md', 'AGENTS.md'] + glob.glob('specs/*.md') + glob.glob('notebooks/*.ipynb')
    return sorted(f for f in files if not f.endswith('R1-01_research_pack.ipynb'))
HASH_BEFORE = {f: sha(f) for f in watched()}
print('Hashed', len(HASH_BEFORE), 'files that must stay unchanged')

In [ ]:
# 3 Create: compute the numbers from data/processed, then write the 3 files
P = 'data/processed/'
shop = pd.read_csv(P + 'shopzilla_contacts_clean.csv', usecols=['channel_name', 'CSAT Score', 'response_hours', 'response_valid', 'team', 'Issue_reported at'])
email = pd.read_csv(P + 'shopzilla_email_clean.csv', usecols=['Customer Remarks'])
tix = pd.read_csv(P + 'tobi_tuning_150.csv')  # tuning set only, the test set stays untouched
n_teams = shop.loc[shop['team'] != 'Other', 'team'].nunique()  # pseudonymised teams, Other not counted

n_all = len(shop)
email_share = (shop['channel_name'] == 'Email').mean()
valid = shop[shop['response_valid'].astype(str).str.lower().eq('true')]  # valid response times only
med = valid.groupby('channel_name')['response_hours'].median()
n_valid = valid.groupby('channel_name').size()
med_text = ', '.join(f'{ch} {med[ch]:.2f} hours (n={n_valid[ch]})' for ch in med.index)
print('Valid response rows:', len(valid), '| per channel:', n_valid.to_dict())
print('Median response_hours per channel:', med.round(2).to_dict())
csat = shop.groupby('channel_name')['CSAT Score'].mean()
csat_text = ', '.join(f'{ch} {v:.2f}' for ch, v in csat.items())
remarks_share = email['Customer Remarks'].notna().mean()
queue_text = ', '.join(f'{q} {n}' for q, n in tix['queue'].value_counts().items())

# KPI baseline numbers (all computed from Shopzilla)
def share45(ch):  # share of CSAT scores 4-5, rows with a score only
    c = shop.loc[shop['channel_name'] == ch, 'CSAT Score'].dropna()
    return round((c >= 4).mean() * 100, 1)
csat45 = {ch: share45(ch) for ch in shop['channel_name'].unique()}
lowest = min(csat45, key=csat45.get)
email_csat45, inb_csat45, out_csat45 = csat45['Email'], csat45['Inbound'], csat45['Outcall']
ev = valid.loc[valid['channel_name'] == 'Email', 'response_hours']
email_median_min = int(round(ev.median() * 60))
email_within24 = round((ev <= 24).mean() * 100, 1)
month = pd.to_datetime(shop['Issue_reported at'], errors='coerce').dt.to_period('M')
email_per_month = format(int(month[shop['channel_name'] == 'Email'].value_counts().max()), ',')
print('KPI baseline:', csat45, '| email median min', email_median_min, '| within 24 h', email_within24, '| per month', email_per_month)

sector = f'''# Sector research: Spreekorb

## Sector and size
- Sector: e-commerce (online retail), Berlin. Spreekorb is an invented company used as a case study.
- Size: about 500 employees (given in the client brief). That makes Spreekorb a large company. In the EU, an SME has fewer than 250 employees. Source: European Commission Recommendation 2003/361/EC.
- Focus: email customer service, SLA (first response time) and CSAT (customer satisfaction).
- CEO Chleo fears AI is a black box, so trust and transparency come first in every proposal.

## Why size matters
- Email volume: many tickets, so a small time saving per email adds up.
- Many agents and teams: a team-level view is useful, but only with a minimum group of 5 people.
- Works council (Betriebsrat): it must be involved in tools that can monitor staff (BetrVG par. 87(1) no. 6).
- GDPR Art. 35 (DPIA): customer emails hold personal data, so a data protection impact assessment is needed before real use.
- EU AI Act (Regulation (EU) 2024/1689): the assessment for worker monitoring is still open (to confirm with a lawyer).
- A large company has the legal, data protection and IT capacity to handle this. A small one often does not.

## Problems (with numbers)
Shopzilla is a public dataset from another company. It is a proxy/benchmark for Spreekorb, not Spreekorb data.
- Email share: {email_share:.1%} of {n_all} contacts are Email (computed from shopzilla_contacts_clean.csv).
- First response time: median per channel is {med_text} (computed from response_hours, valid rows only).
- CSAT: mean CSAT per channel is {csat_text}; {lowest} has the lowest share of CSAT scores 4-5 at {csat45[lowest]}% (computed from CSAT Score in shopzilla_contacts_clean.csv).
- Free text: {remarks_share:.1%} of Email contacts have a customer remark (computed from shopzilla_email_clean.csv), so root cause work has limited text.
- Ticket mix: tickets per queue in the Tobi-Bueck tuning sample are {queue_text} (computed from tobi_tuning_150.csv only, the test set is not used).
- Spreekorb own volume, SLA target and CSAT: not available, source to find (Ayhan).

## Email KPI baseline
| KPI | Market benchmark | Spreekorb target (proposed) | Spreekorb today | Basis |
|---|---|---|---|---|
| Email first response (SLA) | German shoppers expect an email reply within 24 h (German shopper expectation, eDesk: https://www.edesk.com/de/blog/5-wesentliche-strategien-fuer-eine-effizientere-bearbeitung-von-kundenanfragen-im-online-handel-deutscher-markt/); e-commerce average response time 17.3 h (E-commerce market average, Netomi 2020: https://ai.netomi.com/hubfs/Content%20Library/customerservice-benchmarkreport-ecommerce.pdf) | ≤ 24 h for 90% of emails, stretch 1 h (target (proposed)) | median 31 h, 58% within 24 h (synthetic) | Shopzilla is chat/phone-paced (email median {email_median_min} min, {email_within24}% within 24 h, computed from Shopzilla), so it is not used as the email SLA baseline; Spreekorb today is synthetic: no triage, manual drafts. |
| Email CSAT (share of 4-5 scores) | e-commerce median 82%, top quartile 92%+ (German market estimate) | 85% (target (proposed)) | {email_csat45}% (computed from Shopzilla email) | Lowest of the channels: Inbound {inb_csat45}%, Outcall {out_csat45}% (computed from Shopzilla). |
| Email volume | 20-50 tickets per 100 orders; another source says 2-5 contacts per 100 orders (German market estimate, sources disagree) | n/a | ~{email_per_month} emails per month (computed from Shopzilla, busiest month) | Sources disagree on contact rate, so volume is used directly, not derived from orders. |
| Unanswered emails | German mystery-mailing study: one in three emails goes unanswered (German market estimate) | 0% (target (proposed)) | 6% reopened or chased again (synthetic) | Pressure signal for UC2. |

Remaining German market estimates (CSAT, contact rate, unanswered emails): source links to be added by Ayhan.

## Data
Shopzilla is a public Kaggle dataset used as a proxy/benchmark for Spreekorb (synthetic status to confirm); it holds contacts, CSAT scores and response times, so it fits the SLA and CSAT problem (https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction). Tobi-Bueck is on Hugging Face, licence CC BY-NC 4.0 (per Hugging Face licence tag), and its multilingual English and German ticket texts with answers fit email triage and reply drafts (https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets). The team pulse is synthetic (made up, not Spreekorb staff) and shows adoption and trust at team level, so it fits the team view without real personal data.
'''

risks = '''# Opportunities and risks

| Opportunity | Risk | Mitigation |
|---|---|---|
| Faster first response with AI reply drafts | A wrong draft is sent to a customer | A human approves every reply before it is sent |
| Root cause insight for the business | The AI invents reasons (hallucination) | LangSmith evals on a held-out test set, cite the source tickets |
| Team-level view of SLA and CSAT | It turns into performance monitoring of people | Team level only, minimum group of 5, Betriebsrat agreement first |
| Use of customer ticket texts | Personal data goes to OpenAI | Data processing agreement (DPA), transfer check, PII masking, GDPR Art. 35 DPIA, EU AI Act (Regulation (EU) 2024/1689) check |
| Trust through transparency (answers CEO Chleo's black box fear) | Staff distrust the tool and do not use it | Show the steps and sources, involve the works council, track the trust pulse |

## One risk avoided on purpose
- No auto-send of replies: the AI only drafts, a human decides.
- No individual agent scoring: no agent names are stored, only team level views.
'''

cases = f'''# Use cases

## UC1 Email triage and reply drafts (human approves)
- Problem it solves: slow first response because agents sort and write every email by hand.
- Data used: Tobi-Bueck tickets (English and German) for tuning and testing.
- KPI: first response time (SLA) and share of drafts approved without edits.
- Human in the loop: an agent reads, edits and approves every draft. Nothing is sent automatically.
- Why it fits a large company: high email volume makes small savings add up, and there is capacity for review and compliance.

## UC2 Root cause report to the business
- Problem it solves: management sees low CSAT or slow replies but not why.
- Data used: Shopzilla contacts, CSAT, response times and customer remarks (proxy for Spreekorb).
- KPI: CSAT trend and the top causes found, checked against source tickets.
- Human in the loop: an analyst checks every cause and its cited tickets before the report goes out.
- Why it fits a large company: enough volume for patterns to show, and several departments (product, logistics, support) can act on the findings.

## UC3 Team-level view
- Problem it solves: leaders cannot see SLA, CSAT and adoption by team.
- Data used: Shopzilla team column ({n_teams} pseudonymised teams, Team NN / Other, computed from shopzilla_contacts_clean.csv, minimum 5 distinct agents per team-month) and the synthetic team pulse.
- KPI: team SLA, team CSAT and the adoption and trust pulse.
- Human in the loop: team leads discuss the view with the works council. Teams under 5 people are never shown.
- Why it fits a large company: many teams make a team view useful, and a Betriebsrat and data protection staff exist to agree the rules.
'''

os.makedirs('research', exist_ok=True)
for path, text in zip(OUT, [sector, risks, cases]):
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(text)
    print('Wrote', path, '(', len(text.splitlines()), 'lines )')

In [ ]:
# 4 Verify: one PASS/FAIL per acceptance item in the spec
RESULTS.clear()
def section(text, title):
    m = re.search(r'^## ' + re.escape(title) + r'[^\n]*\n(.*?)(?=^## |\Z)', text, re.S | re.M)
    return m.group(1) if m else ''

check('All 3 output files exist', all(os.path.exists(f) for f in OUT))
s = open(OUT[0], encoding='utf-8').read()
r = open(OUT[1], encoding='utf-8').read()
u = open(OUT[2], encoding='utf-8').read()

# acceptance 1: sector, size (500 = large), why size matters
check('Sector, size (500 = large) and why size matters', 'e-commerce' in s and '500' in s and 'large company' in s and '2003/361/EC' in s and section(s, 'Why size matters').strip() != '' and 'Betriebsrat' in s)
check('Legal names present (GDPR Art. 35, EU AI Act 2024/1689)', 'GDPR Art. 35' in s and '2024/1689' in s and 'GDPR Art. 35' in r)

# acceptance 2: every line with a digit is computed or an estimate (or the one EU citation)
prob = section(s, 'Problems')
bad = [l for l in prob.splitlines() if re.search(r'\d', l) and not any(k in l for k in ('estimate', 'computed from', '2003/361/EC'))]
if bad:
    print('  lines without a marker:', bad)
check('Problems: every line with a digit has estimate / computed from / 2003/361/EC', prob.strip() != '' and not bad)
check('Problems: at least 3 numbers computed from data', prob.count('computed from') >= 3)

# email KPI baseline: section after Problems, 4 KPI rows, every row with a digit is labelled
kpi = section(s, 'Email KPI baseline')
kpi_rows = [l for l in kpi.splitlines() if l.startswith('|')][2:]  # skip header and separator
labels = ('German market estimate', 'computed from Shopzilla', 'synthetic', 'target (proposed)', 'eDesk', 'Netomi 2020')
check('Email KPI baseline section exists after Problems with 4 KPI rows', s.find('## Problems') < s.find('## Email KPI baseline') < s.find('## Data') and len(kpi_rows) == 4)
unlabelled = [l[:40] for l in kpi_rows if re.search(r'\d', l) and not any(k in l for k in labels)]
if unlabelled:
    print('  rows without a label:', unlabelled)
check('Email KPI baseline: every row with a digit has a label', len(kpi_rows) > 0 and not unlabelled)
sla_urls = ('https://www.edesk.com/de/blog/5-wesentliche-strategien-fuer-eine-effizientere-bearbeitung-von-kundenanfragen-im-online-handel-deutscher-markt/', 'https://ai.netomi.com/hubfs/Content%20Library/customerservice-benchmarkreport-ecommerce.pdf')
check('SLA row contains both source URLs, 24 h and 17.3 h', len(kpi_rows) > 0 and all(x in kpi_rows[0] for x in sla_urls) and '24 h' in kpi_rows[0] and '17.3 h' in kpi_rows[0])

# acceptance 3: data in 2-3 sentences with links
data = section(s, 'Data').strip()
sentences = [x for x in re.split(r'(?<=[.!?])\s+', data) if x]
urls = ('https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction', 'https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets')
pulse = data.split('The team pulse')[-1]  # the pulse sentence
check('Data described in 2-3 sentences (what, where from, why it fits)', 2 <= len(sentences) <= 3 and all(k in data for k in ('Kaggle', 'Hugging Face', 'fits')))
check('Data has both URLs and the pulse sentence says synthetic', all(x in data for x in urls) and 'synthetic' in pulse)

# acceptance 4: opportunity/risk table + one risk avoided on purpose
rows = [l for l in r.splitlines() if l.startswith('|')]
check('Opportunity/risk table has at least 4 data rows', len(rows) - 2 >= 4)
check('One risk avoided on purpose is present', 'one risk avoided on purpose' in r.lower() and 'auto-send' in r and 'individual agent' in r)

# acceptance 5: 2-3 use cases, each with a why-it-fits line
n_uc = len(re.findall(r'^## UC\d', u, re.M))
n_why = u.count('Why it fits a large company')
check('2-3 use cases, each with a Why it fits a large company line', 2 <= n_uc <= 3 and n_why == n_uc)
check('Each use case names problem, data, KPI and human in the loop', all(u.count(k) == n_uc for k in ('Problem it solves', 'Data used', 'KPI', 'Human in the loop')))
print('Verify:', sum(RESULTS), 'PASS,', len(RESULTS) - sum(RESULTS), 'FAIL')

In [ ]:
# 5 Unchanged: files we do not own must have the same hash as before
after = {f: sha(f) for f in watched()}
changed = [f for f in HASH_BEFORE if after.get(f) != HASH_BEFORE[f]]
check('Watched files unchanged (' + str(len(HASH_BEFORE)) + ' files)', not changed and set(after) == set(HASH_BEFORE))
if changed:
    print('  changed:', changed)
print('Overall (verify + unchanged):', sum(RESULTS), 'PASS of', len(RESULTS))

In [ ]:
# 6 Undo: restore the backup. Only runs if RUN_UNDO = True
RUN_UNDO = False
if RUN_UNDO:
    for f in OUT:
        if f in EXISTED:
            shutil.copy2(os.path.join(BACKUP, f), f)  # put the old file back
            print('Restored', f)
        elif os.path.exists(f):
            os.remove(f)  # file did not exist before, remove it
            print('Removed', f)
else:
    print('RUN_UNDO is False, nothing restored')

In [ ]:
# 7 Git: print the commands only, Ayhan runs them
print('git status')
print('git add research/sector_research.md research/opportunities_risks.md research/use_cases.md notebooks/R1-01_research_pack.ipynb')
print('git commit -m "R1-01 research pack"')